For apples-to-apples comparison, we reimplement SATURN.

Our implementation is faithful to the original while being more concise and accurate.

# Import

In [1]:
import time
import json
import math
import pickle
import random
import numpy as np
import pandas as pd
import scanpy as sc

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from pytorch_metric_learning.losses import ArcFaceLoss,SubCenterArcFaceLoss

from tqdm import tqdm
#from tqdm.notebook import tqdm
from scipy.stats import rankdata
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score,f1_score

device=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Model components

In [2]:
def full_block(in_features,out_features,dropout_rate=0):
    return nn.Sequential(
        nn.Linear(in_features,out_features),
        nn.LayerNorm(out_features),
        nn.ReLU(),
        nn.Dropout(p=dropout_rate),
    )

# Model

In [3]:
# serve as both pretrain and metric train model, switch via the ‘metric_learning_mode’ variable
class SATURN_Model(torch.nn.Module):
    def __init__(self,gene_centroid_scores_dict,hid_dim,emb_dim,fix_g_to_m_weights=True,dropout_rate=0):
        super().__init__()
        self.metric_learning_mode=False
        
        self.dropout_rate=dropout_rate
        self.initial_weights=nn.ParameterDict()
        self.gene_to_macrogene_weights=nn.ParameterDict()
        for species in gene_centroid_scores_dict.keys():
            W0=gene_centroid_scores_dict[species].float().t()
            self.initial_weights[species]=nn.Parameter(W0,requires_grad=False)
            if fix_g_to_m_weights is True:
                self.gene_to_macrogene_weights[species]=nn.Parameter(W0.clone(),requires_grad=False)
            else:
                self.gene_to_macrogene_weights[species]=nn.Parameter(W0.clone())

            self.num_macrogenes=gene_centroid_scores_dict[species].shape[1] # num_macrogenes should be same for all species

        self.species_to_idx={species:idx for idx,species in enumerate(gene_centroid_scores_dict.keys())}
        self.num_species=len(self.species_to_idx.keys())
        
        self.layer_norm=nn.LayerNorm(self.num_macrogenes)
        self.encoder=nn.Sequential(
            full_block(self.num_macrogenes,hid_dim,dropout_rate),
            full_block(hid_dim,emb_dim,dropout_rate)
        )

        self.decoder=nn.Sequential(
            full_block(emb_dim+len(self.species_to_idx),hid_dim,dropout_rate),
            nn.Linear(hid_dim, self.num_macrogenes),
            nn.Softplus()
        )

    def forward(self,gene_expressions,species):
        w=F.softplus(self.gene_to_macrogene_weights[species]) #softplus(x)=log(1+exp(x))
        # We use softplus instead of log_when_init‑exp_when_forward to enforce positivity of the weights during forward,
        # avoiding the inappropriate scaling introduced by log‑exp (since the initial weights are already log‑transformed ranks).
        x=nn.functional.linear(gene_expressions,w)
        x=self.layer_norm(x)
        x=F.relu(x)
        x=F.dropout(x,self.dropout_rate)

        encoder_input=x # encoder_input is "macrogene expression"
        encoded=self.encoder(encoder_input)

        if self.metric_learning_mode is True:
            return encoded

        species_idx=self.species_to_idx[species]
        species_idx=torch.tensor(species_idx).to(encoded.device)
        species_onehot=F.one_hot(species_idx,num_classes=self.num_species).to(encoded.dtype)
        one_hot_expanded=species_onehot.unsqueeze(0).expand(encoded.shape[0],-1)
        decoder_input=torch.cat([encoded,one_hot_expanded],dim=-1)

        decoded=self.decoder(decoder_input)
        return encoder_input,encoded,decoded

    def ae_loss(self,pred,y):
        loss=F.mse_loss(pred,y)
        return loss

    def weights_to_original_distance_loss(self,W,species):
        diff=W-self.initial_weights[species]
        loss=torch.mean(diff**2)
        return loss
    # WE replace the 'gene_weight_ranking_loss' with 'weights_to_original_distance_loss',
    # both loss aim to prevent the macrogenes from becoming meaningless to oversimplify the autoencoding task, but the latter is more stable.

    #++ Later, WE find that:
    # just fix the gene_to_macrogene_weights, rather than restrict its changes, would be best for result embeddings.

# Dataset with "macrogenes"/centroids

In [4]:
class SATURN_SpeciesDataset(torch.utils.data.Dataset):
    def __init__(self,species_info):
        self.X=species_info['X']
        self.embedding_ref=species_info['embedding_ref']
        self.cell_names=species_info['cell_names']
        self.gene_names=species_info['gene_names']
        self.labels=species_info['labels']
        self.category_to_id=species_info['category_to_id']
        self.id_to_category=dict(zip(self.category_to_id.values(),self.category_to_id.keys()))
        
        self.gene_centroid_scores=None
        self.len=len(self.labels)

    def __getitem__(self,index):
        R_dict={}
        R_dict['cell_name']=self.cell_names[index]
        R_dict['X']=self.X[index].coalesce().to_dense()
        R_dict['label']=self.labels[index]
        R_dict['cell_type']=self.id_to_category[R_dict['label'].item()]
        return R_dict

    def __len__(self):
        return self.len

In [5]:
def default_centroids_scores(dd):
    ranked=rankdata(dd,axis=1) # rank 1 is close rank 500 is far
    to_scores=np.log1p(1/ranked) # log 1 is close log 1/500 is far
    to_scores=((to_scores)**2)*2
    return to_scores

def make_centroids(embeds,num_centroids,seed=42):
    print(f"Making {num_centroids} Centroids for mission {mission_name}")
    kmeans_obj=KMeans(n_clusters=num_centroids,random_state=seed,n_init='auto').fit(embeds)
    dd=kmeans_obj.transform(embeds) # dd is distance frome each gene to centroid
    gene_centroid_scores=default_centroids_scores(dd)
    return gene_centroid_scores

# For train and test

In [6]:
def SATURN_test(model,test_loaders,device,ArcFace_loss_function=None):
    if model.metric_learning_mode is False:
        to_return_metrics={'ae_loss':{},'wto_loss':{}}
    else:
        to_return_metrics={'ArcFace_loss':{},'wto_loss':{},'accuracy':{},'f1_score':{}}

    model.eval()
    with torch.no_grad():
        for species,test_loader in test_loaders:
            species_total_main_loss=0
            species_total_steps=0
            all_labels,all_preds,all_probs=[],[],[]
            for data in tqdm(test_loader,desc=f"Testing {species}",leave=False):
                gene_expressions=data['X'].to(torch.float32).to(device)
                if model.metric_learning_mode is False:
                    loss_type='ae'
                    encoder_input,encoded,decoded=model(gene_expressions,species)
                    main_loss=model.ae_loss(decoded,encoder_input)
                else:
                    loss_type='ArcFace'
                    ArcFace_loss_function.eval()
                    labels=data['label'].to(torch.int64)
                    encoded=model(gene_expressions,species)
                    embeddings=nn.functional.normalize(encoded,p=2,dim=-1)
                    main_loss=ArcFace_loss_function[species](embeddings,labels)

                    logits=ArcFace_loss_function[species].get_cosine(embeddings)*ArcFace_loss_function[species].scale
                    probs=torch.softmax(logits,dim=1)
                    _,preds=torch.max(probs,1)
                    all_labels.append(labels.cpu().numpy())
                    all_preds.append(preds.cpu().numpy())
                    all_probs.append(probs.cpu().numpy())    
                current_batch_size=gene_expressions.shape[0]
                species_total_main_loss+=main_loss.item()*current_batch_size
                species_total_steps+=current_batch_size

            mean_main_loss=species_total_main_loss/species_total_steps
            wto_loss=model.weights_to_original_distance_loss(model.gene_to_macrogene_weights[species],species)
            to_return_metrics[f'{loss_type}_loss'][species]=round(mean_main_loss,4)
            to_return_metrics['wto_loss'][species]=round(wto_loss.item(),4)

            print(f'Species: {species}')
            print(f'{loss_type}_loss:{mean_main_loss:.4f}, wto_loss:{wto_loss:.4f}')
            
            if model.metric_learning_mode is True:
                all_labels=np.concatenate(all_labels)
                all_preds=np.concatenate(all_preds)
                all_probs=np.concatenate(all_probs)
                accuracy=accuracy_score(all_labels,all_preds)
                f1=f1_score(all_labels,all_preds,average='weighted')

                to_return_metrics['accuracy'][species]=round(accuracy,4)
                to_return_metrics['f1_score'][species]=round(f1,4)
                print('accuracy:{:.4f}, f1_score:{:.4f}'.format(accuracy,f1))
        torch.cuda.empty_cache()
        return to_return_metrics

In [7]:
#autoencoder pretrain or metric train, depends on whether model.metric_learning_mode is False or True
def SATURN_train(model,training_config,train_loaders,val_loaders,optimizer,device,epochs,ArcFace_loss_function=None):
    metrics_by_epoch={}
    torch.cuda.empty_cache()
    for epoch in range(epochs):
        total_loss,total_main_loss,total_wto_loss=0,0,0
        model.train()
        species_list=[species for species,dl in train_loaders]
        iterators=[iter(dl) for species,dl in train_loaders]
        pseudo_combined_batch_steps=len(species_list)

        total_steps=sum([len(dl) for species,dl in train_loaders])
        progress_bar=tqdm(total=total_steps,desc=f"Epoch {epoch+1}")
        i=0
        while True:
            selected_idx=i% pseudo_combined_batch_steps
            try:
                data=next(iterators[selected_idx])
            except StopIteration:
                break

            species=species_list[selected_idx]
            gene_expressions=data['X'].to(torch.float32).to(device)
            if model.metric_learning_mode is False:
                loss_type='ae'
                encoder_input,encoded,decoded=model(gene_expressions,species)
                main_loss=model.ae_loss(decoded,encoder_input)
            else:
                loss_type='ArcFace'
                ArcFace_loss_function.train()
                labels=data['label'].to(torch.int64)
                encoded=model(gene_expressions,species)
                embeddings=nn.functional.normalize(encoded,p=2,dim=-1)
                main_loss=ArcFace_loss_function[species](embeddings,labels)
            if training_config['wto_penalty']==0:
                wto_loss=0
            else:
                wto_loss=model.weights_to_original_distance_loss(model.gene_to_macrogene_weights[species],species)
            loss=(main_loss +training_config['wto_penalty']*wto_loss)/pseudo_combined_batch_steps
            loss.backward()

            i+=1
            if i%pseudo_combined_batch_steps==0:
                optimizer.step()
                optimizer.zero_grad()
            total_loss+=loss.item()*pseudo_combined_batch_steps
            total_main_loss+=main_loss.item()
            if training_config['wto_penalty']!=0:
                total_wto_loss+=wto_loss.item()
            progress_bar.update(1)
            progress_bar.set_postfix(loss=total_loss/progress_bar.n)
        progress_bar.close()
        print(f'epoch {epoch+1}, loss:{total_loss/total_steps:.4f},\
        {loss_type}_loss:{total_main_loss/total_steps:.4f}, wto_loss:{total_wto_loss/total_steps:.4f}')

        if epoch%5==4:
            print('At epoch '+str(epoch+1),':')
            metrics_by_epoch[epoch+1]=SATURN_test(model,val_loaders,device,ArcFace_loss_function)
            torch.save(model.state_dict(),'./model/smodel_'+str(epoch+1)+'_'+str(int(time.time()))+'.pkl')
            if model.metric_learning_mode is True:
                torch.save(ArcFace_loss_function.state_dict(),'./model/sloss_func_'+str(epoch+1)+'_'+str(int(time.time()))+'.pkl')
    with open('./model/metrics_by_epoch_'+str(int(time.time()))+'.json','w') as f:
        json.dump(metrics_by_epoch,f,indent=4)

# Train and Test

For fair comparison, we use the exact same dataset for SATURN as was used for GECross:

the 'Frog_Zebrafish_2000hv_60000cell' dataset, being built in 'paint_cell_image_representations.ipynb'.

## dataloader

In [8]:
#dataset building
#let REBuild=0 when you have already created the dataset
REBuild=0

mission_name='Frog_Zebrafish_2000hv_60000cell'

embedding_type='ESM1b'
all_species=['frog','zebrafish']

SATURN_train_set,SATURN_val_set,SATURN_test_set={},{},{}
if REBuild==1:
    SATURN_dataset={}
    all_embeddings=torch.tensor([])
    for species in all_species:
        with open(f'./use_data/Save_DataSet/{mission_name}/{embedding_type}_{species}_process_return.pkl','rb') as f:
            process_return=pickle.load(f)
        SATURN_dataset[species]=SATURN_SpeciesDataset(process_return)
        all_embeddings=torch.cat((all_embeddings,SATURN_dataset[species].embedding_ref),dim=0)
    
    all_gene_centroid_scores=make_centroids(all_embeddings,num_centroids=500) # 500 is 1/4 of the num of hv_genes:2000

    loc_pointer=0
    for species in all_species:
        SATURN_dataset[species].gene_centroid_scores=torch.tensor(all_gene_centroid_scores[loc_pointer:loc_pointer+len(SATURN_dataset[species].embedding_ref)])
        loc_pointer+=len(SATURN_dataset[species].embedding_ref)

        species_train_size=int(len(SATURN_dataset[species])*0.6)
        species_val_size=int(len(SATURN_dataset[species])*0.2)
        species_test_size=len(SATURN_dataset[species])-species_train_size-species_val_size
        SATURN_train_set[species],SATURN_val_set[species],SATURN_test_set[species]=torch.utils.data.random_split(SATURN_dataset[species],[species_train_size,species_val_size,species_test_size])

        for SATURN_set,set_type in [(SATURN_train_set,'train'),(SATURN_val_set,'val'),(SATURN_test_set,'test')]:
            with open(f'./use_data/Save_DataSet/{mission_name}/SATURN_{embedding_type}_{species}_{set_type}_set.pkl','wb') as f:
                pickle.dump(SATURN_set[species],f)
else:
    for species in all_species:
        for SATURN_set,set_type in [(SATURN_train_set,'train'),(SATURN_val_set,'val'),(SATURN_test_set,'test')]:
            with open(f'./use_data/Save_DataSet/{mission_name}/SATURN_{embedding_type}_{species}_{set_type}_set.pkl','rb') as f:
                SATURN_set[species]=pickle.load(f)

num_classes_of_species={}
for species in all_species:
    num_classes_of_species[species]=len(SATURN_train_set[species].dataset.category_to_id)

In [9]:
#dataloader building
all_species=['frog','zebrafish']

gene_centroid_scores_dict={}
for species in all_species:
    gene_centroid_scores_dict[species]=SATURN_train_set[species].dataset.gene_centroid_scores

train_loaders,val_loaders,test_loaders=[],[],[]
for species in all_species:
    for _loaders,_set,_type in [(train_loaders,SATURN_train_set,'train'),(val_loaders,SATURN_val_set,'val'),(test_loaders,SATURN_test_set,'test')]:
        _loader=DataLoader(dataset=_set[species],batch_size=1024,shuffle=True,drop_last=(_type=='train'))
        _loaders.append((species,_loader))

## pretrain

In [18]:
hid_dim=128
emb_dim=256
fix_g_to_m_weights=True
dropout_rate=0.1
model=SATURN_Model(gene_centroid_scores_dict,hid_dim,emb_dim,fix_g_to_m_weights,dropout_rate).to(device)
model.metric_learning_mode=False

optimizer=optim.Adam(model.parameters(),lr=0.0005)

epochs=100
training_config={'wto_penalty':0} #since gene to macrogene weights are fixed
SATURN_train(model,training_config,train_loaders,val_loaders,optimizer,device,epochs)

Epoch 1: 100%|█████████████████████████████████████████████████████████████| 70/70 [05:41<00:00,  4.88s/it, loss=0.707]


epoch 1, loss:0.7070,        ae_loss:0.7070, wto_loss:0.0000


Epoch 2: 100%|█████████████████████████████████████████████████████████████| 70/70 [05:53<00:00,  5.06s/it, loss=0.461]


epoch 2, loss:0.4607,        ae_loss:0.4607, wto_loss:0.0000


Epoch 3: 100%|██████████████████████████████████████████████████████████████| 70/70 [05:56<00:00,  5.09s/it, loss=0.37]


...

epoch 99, loss:0.0023,        ae_loss:0.0023, wto_loss:0.0000


Epoch 100: 100%|█████████████████████████████████████████████████████████| 70/70 [06:05<00:00,  5.22s/it, loss=0.00223]


epoch 100, loss:0.0022,        ae_loss:0.0022, wto_loss:0.0000
At epoch 100 :


Species: frog
ae_loss:0.0007, wto_loss:0.0000


Species: zebrafish
ae_loss:0.0010, wto_loss:0.0000


## pretest

In [27]:
hid_dim=128
emb_dim=256
fix_g_to_m_weights=True
model=SATURN_Model(gene_centroid_scores_dict,hid_dim,emb_dim,fix_g_to_m_weights).to(device)
model.load_state_dict(torch.load("./model/s_gtm_weights_fixed_25_epoch_pretrain_model.pkl",weights_only=True))
model.metric_learning_mode=False

_=SATURN_test(model,test_loaders,device)

Species: frog
ae_loss:0.0342, wto_loss:0.0000


Species: zebrafish
ae_loss:0.0525, wto_loss:0.0000


## metric learning with ArcFaceLoss

### metric train

In [10]:
hid_dim=128
emb_dim=256
fix_g_to_m_weights=True
dropout_rate=0.1
model=SATURN_Model(gene_centroid_scores_dict,hid_dim,emb_dim,fix_g_to_m_weights,dropout_rate).to(device)
model.load_state_dict(torch.load("./model/s_gtm_weights_fixed_25_epoch_pretrain_model.pkl",weights_only=True))
model.metric_learning_mode=True

margin=11.5 # np.radians(11.5)≈0.2
scale=32
num_sub_centers={'frog':5,'zebrafish':3}
loss_function=nn.ModuleDict({
    species: SubCenterArcFaceLoss(sub_centers=num_sub_centers[species],num_classes=num_classes,embedding_size=emb_dim,margin=margin,scale=scale)
    for species,num_classes in num_classes_of_species.items()
}).to(device)

all_params=list(model.parameters())
all_params.extend(list(loss_function.parameters()))
optimizer=optim.Adam(all_params,lr=0.001)

epochs=100
training_config={'wto_penalty':0}
SATURN_train(model,training_config,train_loaders,val_loaders,optimizer,device,epochs,loss_function)

Epoch 1: 100%|██████████████████████████████████████████████████████████████| 70/70 [05:53<00:00,  5.05s/it, loss=8.84]


epoch 1, loss:8.8403,        ArcFace_loss:8.8403, wto_loss:0.0000


Epoch 2: 100%|██████████████████████████████████████████████████████████████| 70/70 [06:02<00:00,  5.18s/it, loss=7.04]


epoch 2, loss:7.0444,        ArcFace_loss:7.0444, wto_loss:0.0000


Epoch 3: 100%|██████████████████████████████████████████████████████████████| 70/70 [06:06<00:00,  5.23s/it, loss=5.94]


...

epoch 99, loss:1.9139,        ArcFace_loss:1.9139, wto_loss:0.0000


Epoch 100: 100%|████████████████████████████████████████████████████████████| 70/70 [05:55<00:00,  5.08s/it, loss=1.93]


epoch 100, loss:1.9316,        ArcFace_loss:1.9316, wto_loss:0.0000
At epoch 100 :


Species: frog
ArcFace_loss:4.2027, wto_loss:0.0000
accuracy:0.7150, f1_score:0.7112


Species: zebrafish
ArcFace_loss:2.6879, wto_loss:0.0000
accuracy:0.8143, f1_score:0.8124


### metric test

In [24]:
hid_dim=128
emb_dim=256
fix_g_to_m_weights=True
model=SATURN_Model(gene_centroid_scores_dict,hid_dim,emb_dim,fix_g_to_m_weights).to(device)
model.load_state_dict(torch.load("./model/s_arcface_model_75_fixed_25_epoch_pretrain.pkl",weights_only=True))
model.metric_learning_mode=True

margin=11.5 # np.radians(11.5)≈0.2
scale=32
num_sub_centers={'frog':5,'zebrafish':3}
loss_function=nn.ModuleDict({
    species: SubCenterArcFaceLoss(sub_centers=num_sub_centers[species],num_classes=num_classes,embedding_size=emb_dim,margin=margin,scale=scale)
    for species,num_classes in num_classes_of_species.items()
}).to(device)
loss_function.load_state_dict(torch.load("./model/s_arcface_loss_func_75_fixed_25_epoch_pretrain.pkl",weights_only=True))

_=SATURN_test(model,test_loaders,device,loss_function)

Species: frog
ArcFace_loss:3.7431, wto_loss:0.0000
accuracy:0.7455, f1_score:0.7410


Species: zebrafish
ArcFace_loss:2.2531, wto_loss:0.0000
accuracy:0.8443, f1_score:0.8414


## or, metric learning with original TripletMarginLoss

### extra prepare

In [10]:
import sys
import os
codes_path=os.path.join(os.getcwd(),'SATURN_codes')
sys.path.insert(0,codes_path)
import losses,miners,distances,reducers

In [11]:
def Triplet_train(model,training_config,train_loaders,loss_func,mining_func,optimizer,device,epochs,ordered_all_species):
    torch.cuda.empty_cache()
    for epoch in range(epochs):
        total_loss,total_triplet_loss,total_wto_loss=0,0,0
        model.train()
        species_list=[species for species,dl in train_loaders]
        iterators=[iter(dl) for species,dl in train_loaders]
        combined_batch_steps=len(species_list)

        total_steps=sum([len(dl) for species,dl in train_loaders])
        progress_bar=tqdm(total=total_steps,desc=f"Epoch {epoch+1}")
        
        embs,labs,spec=[],[],[]
        combined_batch_wto_loss=0
        i=0
        while True:
            optimizer.zero_grad()
            selected_idx=i% combined_batch_steps
            try:
                data=next(iterators[selected_idx])
            except StopIteration:
                break

            species=species_list[selected_idx]
            species_index=np.argmax(np.array(ordered_all_species)==species)

            gene_expressions=data['X'].to(torch.float32).to(device)
            labels=data['label'].to(torch.int64)
            labels+=55*species_index # works if there is less than 1000 classes per species
            # as the miner needs unique label for all cross-species classes

            encoded=model(gene_expressions,species)
            embeddings=nn.functional.normalize(encoded,p=2,dim=-1)
            
            embs.append(embeddings)
            labs.append(labels)
            spec.append(species_index*torch.ones_like(labels))

            combined_batch_wto_loss+=model.weights_to_original_distance_loss(model.gene_to_macrogene_weights[species],species)/combined_batch_steps

            i+=1
            if i%combined_batch_steps==0:
                embs=torch.cat(embs).to(device)
                labs=torch.cat(labs).to(device)
                spec=torch.cat(spec).to(device)

                indices_tuple=mining_func(embs,labs,spec,mnn=True)#mnn - use mutual nearest neighbors for metric learning mining
                triplet_loss=loss_func(embs,labs,indices_tuple)

                equalize_triplets_species=True
                if equalize_triplets_species:
                    species_mapped=[spec[i] for i in indices_tuple] # a,p,n species vectors
                    a_spec=species_mapped[0]
                    p_spec=species_mapped[1]
                    a_uq,a_inv,a_ct=torch.unique(a_spec,return_counts=True,return_inverse=True)
                    p_uq,p_inv,p_ct=torch.unique(p_spec,return_counts=True,return_inverse=True)

                    a_prop=a_ct/torch.sum(a_ct) # Proportions of total ie 1/4
                    p_prop=p_ct/torch.sum(p_ct) # Proportions of total ie 3/4
                    a_balance=torch.reciprocal(a_prop)/len(a_prop) # balancing ie * 4 / 1, then divide by num species
                    p_balance=torch.reciprocal(p_prop)/len(p_prop) # balancing ie 4 / 3, then divide by num species
                    a_bal_inv=a_balance[a_inv]
                    p_bal_inv=p_balance[p_inv]
                    triplet_loss=torch.mul(torch.mul(triplet_loss,a_bal_inv),p_bal_inv).mean()

                loss=triplet_loss +training_config['wto_penalty']*combined_batch_wto_loss
                loss.backward()
                optimizer.step()

                total_loss+=loss.item()
                total_triplet_loss+=triplet_loss.item()
                total_wto_loss+=combined_batch_wto_loss.item()
                combined_batch_wto_loss=0
                embs,labs,spec=[],[],[]
                progress_bar.update(1)
                progress_bar.set_postfix(loss=total_loss/progress_bar.n*combined_batch_steps)
            else:
                progress_bar.update(1)
                
        progress_bar.close()
        print(f'epoch {epoch+1}, loss:{total_loss*combined_batch_steps/total_steps:.4f},\
            triplet_loss:{total_triplet_loss*combined_batch_steps/total_steps:.4f}, \
            wto_loss:{total_wto_loss*combined_batch_steps/total_steps:.4f}')

        if epoch%5==4:
            print('At epoch '+str(epoch+1),':')
            torch.save(model.state_dict(),'./model/s_triplet_model_'+str(epoch+1)+'_'+str(int(time.time()))+'.pkl')

### metric train

In [12]:
hid_dim=128
emb_dim=256
fix_g_to_m_weights=True

dropout_rate=0.1
model=SATURN_Model(gene_centroid_scores_dict,hid_dim,emb_dim,fix_g_to_m_weights,dropout_rate).to(device)
model.load_state_dict(torch.load("./model/s_gtm_weights_fixed_25_epoch_pretrain_model.pkl",weights_only=True))
model.metric_learning_mode=True

optimizer=optim.Adam(model.parameters(),lr=1e-3)
distance=distances.CosineSimilarity()
reducer=reducers.ThresholdReducer(low=0)
loss_func=losses.TripletMarginLoss(margin=0.2,distance=distance,reducer=reducer)
mining_func=miners.TripletMarginMiner(margin=0.2,distance=distance,type_of_triplets="semihard",miner_type="cross_species")

epochs=100
training_config={'wto_penalty':0} #since gene to macrogene weights are fixed
ordered_all_species=['frog','zebrafish']
Triplet_train(model,training_config,train_loaders,loss_func,mining_func,optimizer,device,epochs,ordered_all_species)

Epoch 1: 100%|█████████████████████████████████████████████████████████████| 70/70 [06:55<00:00,  5.93s/it, loss=0.104]


epoch 1, loss:0.1043,            triplet_loss:0.1043,             wto_loss:0.0000


Epoch 2: 100%|████████████████████████████████████████████████████████████| 70/70 [05:32<00:00,  4.75s/it, loss=0.0989]


epoch 2, loss:0.0989,            triplet_loss:0.0989,             wto_loss:0.0000


Epoch 3: 100%|████████████████████████████████████████████████████████████| 70/70 [05:13<00:00,  4.47s/it, loss=0.0978]


...

epoch 98, loss:0.0869,            triplet_loss:0.0869,             wto_loss:0.0000


Epoch 99: 100%|████████████████████████████████████████████████████████████| 70/70 [07:05<00:00,  6.08s/it, loss=0.086]


epoch 99, loss:0.0860,            triplet_loss:0.0860,             wto_loss:0.0000


Epoch 100: 100%|██████████████████████████████████████████████████████████| 70/70 [07:06<00:00,  6.10s/it, loss=0.0868]

epoch 100, loss:0.0868,            triplet_loss:0.0868,             wto_loss:0.0000
At epoch 100 :


# For Get embeddings

In [12]:
def get_embeddings(model,test_loaders,device,sub_center_weights=None,num_sub_centers=None,label_celltype_dict=None):
    X_data=[]
    obs_df_data=[]
    model.eval()
    with torch.no_grad():
        for species,test_loader in test_loaders:
            for data in tqdm(test_loader,desc=f"Getting embeddings for {species}",leave=False):
                gene_expressions=data['X'].to(torch.float32).to(device)

                encoded=model(gene_expressions,species)
                embeddings=nn.functional.normalize(encoded,p=2,dim=-1).cpu().numpy()
                for i in range(len(data['X'])):
                    X_data.append(embeddings[i])
                    obs_df_data.append([data['cell_name'][i],species,data['cell_type'][i]])

            if sub_center_weights is not None:
                s_weights=sub_center_weights[f'{species}.W'].T
                s_weights=nn.functional.normalize(s_weights,p=2,dim=-1).cpu().numpy()
                for i,subcenter_embedding in enumerate(s_weights):
                    X_data.append(subcenter_embedding)
    
                    label=math.floor(i/num_sub_centers[species])
                    cell_type=f'{label_celltype_dict[species][label]}_subcenter'
                    k_th=i%num_sub_centers[species]
                    sub_center_name=f'{species}_{cell_type}_{k_th}'
                    obs_df_data.append([sub_center_name,species,cell_type])
            print(f'Done get embeddings for {species}')
        torch.cuda.empty_cache()

    obs_df=pd.DataFrame(obs_df_data,columns=['cell_name','species','cell_type'])
    obs_df=obs_df.set_index('cell_name')
    adata=sc.AnnData(X=np.array(X_data),obs=obs_df)    
    return adata

# Get embeddings

## get embeddings from model trained by ArcFace metric

In [14]:
all_species=['frog','zebrafish']
label_celltype_dict={}
for species in all_species:
    label_celltype_dict[species]=SATURN_train_set[species].dataset.id_to_category

In [32]:
#use only test loaders
hid_dim=128
emb_dim=256
fix_g_to_m_weights=True
model=SATURN_Model(gene_centroid_scores_dict,hid_dim,emb_dim,fix_g_to_m_weights).to(device)
model.load_state_dict(torch.load("./model/s_arcface_model_40_fixed_25_epoch_pretrain.pkl",weights_only=True))
model.metric_learning_mode=True

sub_center_weights=torch.load("./model/s_arcface_loss_func_40_fixed_25_epoch_pretrain.pkl",weights_only=True)

num_sub_centers={'frog':5,'zebrafish':3}
embedding_adata=get_embeddings(model,test_loaders,device,sub_center_weights,num_sub_centers,label_celltype_dict)
embedding_adata.write('./use_data/Save_for_Drawing/s_arcface_model_40_fixed_25_epoch_pretrain_embedding.h5ad')

Done get embeddings for frog


Done get embeddings for zebrafish


In [20]:
#use only test loaders
hid_dim=128
emb_dim=256
fix_g_to_m_weights=True
model=SATURN_Model(gene_centroid_scores_dict,hid_dim,emb_dim,fix_g_to_m_weights).to(device)
model.load_state_dict(torch.load("./model/s_arcface_model_75_fixed_25_epoch_pretrain.pkl",weights_only=True))
model.metric_learning_mode=True

sub_center_weights=torch.load("./model/s_arcface_loss_func_75_fixed_25_epoch_pretrain.pkl",weights_only=True)

num_sub_centers={'frog':5,'zebrafish':3}
embedding_adata=get_embeddings(model,test_loaders,device,sub_center_weights,num_sub_centers,label_celltype_dict)
embedding_adata.write('./use_data/Save_for_Drawing/s_arcface_model_75_fixed_25_epoch_pretrain_embedding.h5ad')

Done get embeddings for frog


Done get embeddings for zebrafish


## get embeddings from model trained by Triplet metric

In [15]:
hid_dim=128
emb_dim=256
fix_g_to_m_weights=True

model=SATURN_Model(gene_centroid_scores_dict,hid_dim,emb_dim,fix_g_to_m_weights).to(device)
model.load_state_dict(torch.load("./model/s_triplet_model_100_fixed_25_epoch_pretrain.pkl",weights_only=True))
model.metric_learning_mode=True

embedding_adata=get_embeddings(model,test_loaders,device)
embedding_adata.write('./use_data/Save_for_Drawing/s_triplet_model_100_fixed_25_epoch_pretrain_embedding.h5ad')

Done get embeddings for frog


Done get embeddings for zebrafish
